# Pre-Market Scanner Backtest

Replica el scanner IBKR en barras 1-min de pre-market (04:00-09:30 ET).

**Preguntas:**
1. ¿A qué hora pre-market se activa la señal en los 87 gaps?
2. ¿Cuánto volumen pre-market es necesario para filtrar ruido?
3. ¿El gap pre-market predice la continuación RTH?
4. ¿Cuántos falsos positivos tiene el trigger pre-market?

**Trigger pre-market (configurable):**
```
vol_premarket_bar / avg_vol_20d > PM_VOL_THRESHOLD
AND price_change_Nmin > PM_PRICE_THRESHOLD
AND hora < 09:30 ET
```


## 1. Config

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

TZ = 'America/New_York'

PM_VOL_THRESHOLD   = 5.0    # vol_bar / avg_vol_20d en pre-market
PM_PRICE_THRESHOLD = 3.0    # % de subida en N barras pre-market
PM_PRICE_WINDOW    = 3      # barras para calcular momentum
FLOAT_MAX          = 50e6   # None = sin filtro

DB_PREMARKET = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_premarket_cache.db')
DB_INTRADAY  = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_intraday_cache.db')
DB_DAILY     = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_daily_cache.db')
FINVIZ_DB    = Path.home() / 'Library/Application Support/finviz-dashboard/finviz_snapshots.db'
FLOAT_CACHE  = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/reactive_signal/edgar_cache/float_cache.json')

print("Config OK")
print(f"  PM_VOL_THRESHOLD  : >{PM_VOL_THRESHOLD}x avg vol 20d")
print(f"  PM_PRICE_THRESHOLD: >{PM_PRICE_THRESHOLD}% en {PM_PRICE_WINDOW} barras")

## 2. Cargar datos

In [ ]:
# ── Pre-market bars ───────────────────────────────────────────────────────────
with sqlite3.connect(DB_PREMARKET) as conn:
    pm_bars = pd.read_sql(
        "SELECT symbol AS ticker, date, dt, open, high, low, close, volume FROM bars",
        conn
    )

pm_bars['bar_ts'] = pd.to_datetime(pm_bars['date'] + 'T' + pm_bars['dt']).dt.tz_localize(TZ, ambiguous='NaT', nonexistent='NaT')
pm_bars['hour']   = pm_bars['bar_ts'].dt.hour
pm_bars['minute'] = pm_bars['bar_ts'].dt.minute
# mso_pm: minutos desde 4:00 AM (inicio pre-market)
pm_bars['mso_pm'] = (pm_bars['hour'] - 4) * 60 + pm_bars['minute']
# Solo barras 4:00-9:29
pm_bars = pm_bars[(pm_bars['hour'] >= 4) & ~((pm_bars['hour'] == 9) & (pm_bars['minute'] >= 30))].copy()
pm_bars = pm_bars.sort_values(['ticker','date','mso_pm']).reset_index(drop=True)

td_pm = pm_bars.groupby(['ticker','date']).ngroups
print(f"Pre-market bars: {len(pm_bars):,}  |  ticker-days: {td_pm}")

# ── RTH bars (para calcular gap real y retorno RTH post-detección) ─────────────
with sqlite3.connect(DB_INTRADAY) as conn:
    rth_bars = pd.read_sql(
        "SELECT symbol AS ticker, date, dt, open, high, low, close, volume FROM bars",
        conn
    )
rth_bars['bar_ts'] = pd.to_datetime(rth_bars['date'] + 'T' + rth_bars['dt']).dt.tz_localize(TZ, ambiguous='NaT', nonexistent='NaT')
rth_bars['mso']    = (rth_bars['bar_ts'].dt.hour - 9) * 60 + rth_bars['bar_ts'].dt.minute - 30
rth_bars = rth_bars[rth_bars['mso'] >= 0].sort_values(['ticker','date','mso']).reset_index(drop=True)
print(f"RTH bars: {len(rth_bars):,}")

# ── Daily bars (avg_vol_20d, prev_close) ──────────────────────────────────────
with sqlite3.connect(DB_DAILY) as conn:
    daily = pd.read_sql(
        "SELECT ticker, bar_date, open, high, low, close, volume FROM daily_bars ORDER BY ticker, bar_date",
        conn
    )
daily = daily.sort_values(['ticker','bar_date']).reset_index(drop=True)
daily['avg_vol_20d'] = daily.groupby('ticker')['volume'].transform(
    lambda x: x.shift(1).rolling(20, min_periods=5).mean()
)
daily['prev_close'] = daily.groupby('ticker')['close'].shift(1)
daily_idx = daily.set_index(['ticker','bar_date'])
print(f"Daily bars: {len(daily):,}")

# ── Finviz bursts ─────────────────────────────────────────────────────────────
with sqlite3.connect(FINVIZ_DB) as conn:
    bursts = pd.read_sql("""
        SELECT ticker, SUBSTR(timestamp,1,10) AS date,
               MIN(timestamp) AS first_ts,
               CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) AS finviz_chg
        FROM snapshots
        WHERE category = 'Top Gainers'
          AND CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) >= 15
        GROUP BY ticker, SUBSTR(timestamp,1,10)
    """, conn)

bursts['first_ts_et'] = pd.to_datetime(bursts['first_ts'], utc=True).dt.tz_convert(TZ)
bursts['mso_finviz']  = (bursts['first_ts_et'].dt.hour - 9) * 60 + bursts['first_ts_et'].dt.minute - 30
bursts = bursts[bursts['mso_finviz'] >= 0].copy()
burst_set = set(zip(bursts['ticker'], bursts['date']))
print(f"Bursts Finviz >=15%: {len(bursts)}")

# ── Float cache ───────────────────────────────────────────────────────────────
import json
float_cache = json.loads(FLOAT_CACHE.read_text()) if FLOAT_CACHE.exists() else {}
print(f"Float cache: {len(float_cache)} tickers")

## 3. Motor del scanner pre-market

In [ ]:
def replay_premarket(ticker, date, pm_bars_df, rth_bars_df, daily_idx, float_cache,
                     vol_threshold=PM_VOL_THRESHOLD,
                     price_threshold=PM_PRICE_THRESHOLD,
                     price_window=PM_PRICE_WINDOW,
                     float_max=FLOAT_MAX):
    """
    Replay barras 1-min pre-market para un ticker+fecha.
    Retorna dict con detección del trigger y métricas de continuación RTH.
    """
    day_pm = pm_bars_df[(pm_bars_df['ticker']==ticker) & (pm_bars_df['date']==date)].copy()
    if day_pm.empty:
        return None

    # avg_vol_20d
    try:
        avg_vol    = daily_idx.loc[(ticker, date), 'avg_vol_20d']
        prev_close = daily_idx.loc[(ticker, date), 'prev_close']
    except KeyError:
        tkr_daily = daily_idx.xs(ticker, level='ticker') if ticker in daily_idx.index.get_level_values('ticker') else None
        if tkr_daily is None: return None
        before = tkr_daily[tkr_daily.index < date]
        if before.empty: return None
        avg_vol    = before.iloc[-1]['avg_vol_20d']
        prev_close = before.iloc[-1]['close']

    if not avg_vol or pd.isna(avg_vol) or avg_vol <= 0:
        return None

    float_shares = float_cache.get(ticker, 0)
    if float_max and float_shares > 0 and float_shares > float_max:
        return {'triggered': False, 'reason': 'float_too_large',
                'ticker': ticker, 'date': date, 'float_shares': float_shares}

    day_pm = day_pm.sort_values('mso_pm').reset_index(drop=True)
    closes  = day_pm['close'].values
    volumes = day_pm['volume'].values
    mso_pms = day_pm['mso_pm'].values   # minutos desde 4:00 AM
    hours   = day_pm['hour'].values
    minutes = day_pm['minute'].values
    n_bars  = len(day_pm)

    # Gap de apertura (primera barra pre-market vs prev_close)
    first_pm_close = closes[0] if n_bars > 0 else prev_close
    gap_open_pct   = (first_pm_close - prev_close) / prev_close * 100 if prev_close else 0

    triggered       = False
    mso_pm_trigger  = None
    time_trigger    = None
    price_trigger   = None
    vol_ratio_trig  = None
    price_chg_trig  = None

    for i in range(price_window, n_bars):
        vol_ratio  = volumes[i] / avg_vol if avg_vol > 0 else 0
        price_chg  = (closes[i] - closes[i - price_window]) / closes[i - price_window] * 100 if closes[i-price_window] > 0 else 0

        if vol_ratio >= vol_threshold and price_chg >= price_threshold:
            triggered      = True
            mso_pm_trigger = int(mso_pms[i])
            time_trigger   = f"{int(hours[i]):02d}:{int(minutes[i]):02d}"
            price_trigger  = float(closes[i])
            vol_ratio_trig = round(vol_ratio, 1)
            price_chg_trig = round(price_chg, 2)
            break

    # Métricas RTH desde apertura (09:30)
    day_rth = rth_bars_df[(rth_bars_df['ticker']==ticker) & (rth_bars_df['date']==date)].copy()
    rth_open  = day_rth.iloc[0]['open']  if not day_rth.empty else None
    rth_close = day_rth.iloc[-1]['close'] if not day_rth.empty else None
    rth_high  = day_rth['high'].max()    if not day_rth.empty else None
    rth_ret_from_pm_trigger = None
    rth_mfe_from_pm_trigger = None

    if triggered and price_trigger and not day_rth.empty:
        rth_ret_from_pm_trigger = round((rth_close - price_trigger) / price_trigger * 100, 2)
        rth_mfe_from_pm_trigger = round((rth_high  - price_trigger) / price_trigger * 100, 2)

    # Gap apertura RTH
    rth_gap_pct = (rth_open - prev_close) / prev_close * 100 if (rth_open and prev_close) else None

    return {
        'ticker': ticker, 'date': date,
        'triggered': triggered,
        'mso_pm_trigger': mso_pm_trigger,    # min desde 4:00 AM
        'time_trigger': time_trigger,
        'price_trigger': price_trigger,
        'vol_ratio': vol_ratio_trig,
        'price_chg_Nm': price_chg_trig,
        'gap_first_pm_pct': round(gap_open_pct, 2),
        'rth_gap_pct': round(rth_gap_pct, 2) if rth_gap_pct else None,
        'rth_ret_from_trigger': rth_ret_from_pm_trigger,
        'rth_mfe_from_trigger': rth_mfe_from_pm_trigger,
        'avg_vol_20d': round(avg_vol),
        'prev_close': round(float(prev_close), 4) if prev_close and not pd.isna(prev_close) else None,
        'float_shares': float_shares,
    }

print("replay_premarket() definida.")

## 4. Ejecutar replay — todos los ticker-days pre-market

In [ ]:
pm_pairs = pm_bars.groupby(['ticker','date']).size().reset_index()[['ticker','date']]
print(f"Ticker-days con pre-market bars: {len(pm_pairs)}")

pm_results = []
for _, row in pm_pairs.iterrows():
    r = replay_premarket(row['ticker'], row['date'], pm_bars, rth_bars, daily_idx, float_cache)
    if r:
        pm_results.append(r)

pm = pd.DataFrame(pm_results)
print(f"Procesados: {len(pm)}")
print(f"  Scanner disparó   : {pm['triggered'].sum()} ({pm['triggered'].mean():.1%})")
print(f"  No disparó        : {(~pm['triggered']).sum()}")

# Merge con bursts Finviz para clasificar TP/FP/Missed
pm_vs_burst = pm.merge(
    bursts[['ticker','date','mso_finviz','finviz_chg']],
    on=['ticker','date'], how='left'
)
pm_triggered  = pm_vs_burst[pm_vs_burst['triggered']==True].copy()
pm_tp         = pm_triggered[pm_triggered['mso_finviz'].notna()]   # detectó Y era burst
pm_fp         = pm_triggered[pm_triggered['mso_finviz'].isna()]    # detectó Y NO era burst
pm_missed     = burst_set - set(zip(pm_tp['ticker'], pm_tp['date']))

n_bursts_pm = len(set(zip(bursts['ticker'], bursts['date'])) & set(zip(pm_pairs['ticker'], pm_pairs['date'])))
precision = len(pm_tp) / len(pm_triggered) * 100 if len(pm_triggered) > 0 else 0
recall    = len(pm_tp) / n_bursts_pm * 100 if n_bursts_pm > 0 else 0

print()
print(f"Bursts con pre-market data: {n_bursts_pm}")
print(f"True Positives : {len(pm_tp)} — Precision: {precision:.1f}%")
print(f"False Positives: {len(pm_fp)}")
print(f"Recall         : {recall:.1f}%")

## 5. ¿A qué hora se activa el trigger? — distribución temporal

In [ ]:
if len(pm_tp) == 0:
    print("Sin TPs — ajustar PM_VOL_THRESHOLD o PM_PRICE_THRESHOLD")
else:
    tp = pm_tp.copy()

    # Hora de detección en ET
    tp['hour_trigger'] = tp['time_trigger'].str[:2].astype(int)
    tp['min_trigger']  = tp['time_trigger'].str[3:5].astype(int)
    tp['mins_before_open'] = (9*60+30) - (tp['hour_trigger']*60 + tp['min_trigger'])

    print("=" * 60)
    print(f"HORA DE DETECCIÓN PRE-MARKET — {len(tp)} True Positives")
    print("=" * 60)
    print(f"\nMinutos antes de apertura RTH:")
    print(f"  Mediana : {tp['mins_before_open'].median():.0f} min antes de 9:30")
    print(f"  Media   : {tp['mins_before_open'].mean():.1f} min")
    print(f"  p25/p75 : {tp['mins_before_open'].quantile(.25):.0f} / {tp['mins_before_open'].quantile(.75):.0f} min")

    print(f"\nDistribución por hora ET:")
    for h in range(4, 10):
        n_h = (tp['hour_trigger'] == h).sum()
        if n_h > 0:
            label = "PRE-RTH" if h < 9 or (h == 9) else "RTH"
            bar = "█" * n_h
            print(f"  {h:02d}:xx ET  {n_h:>3}  {bar}")

    print(f"\nDetectados antes de 8:00 ET (3+ h antes): {(tp['hour_trigger'] < 8).sum()}")
    print(f"Detectados 8:00-9:00 ET (1-1.5h antes):   {((tp['hour_trigger'] >= 8) & (tp['hour_trigger'] < 9)).sum()}")
    print(f"Detectados 9:00-9:30 ET (<30 min antes):  {((tp['hour_trigger'] == 9) & (tp['min_trigger'] < 30)).sum()}")

    print(f"\nGap primer bar pre-market vs prev_close:")
    print(f"  Mediana : +{tp['gap_first_pm_pct'].median():.1f}%")
    print(f"  Media   : +{tp['gap_first_pm_pct'].mean():.1f}%")
    print(f"  <5%: {(tp['gap_first_pm_pct'] < 5).sum()}  5-15%: {((tp['gap_first_pm_pct']>=5)&(tp['gap_first_pm_pct']<15)).sum()}  15-50%: {((tp['gap_first_pm_pct']>=15)&(tp['gap_first_pm_pct']<50)).sum()}  >50%: {(tp['gap_first_pm_pct']>=50).sum()}")

    print(f"\nRetorno RTH desde precio de trigger (pre-market):")
    has_rth = tp['rth_ret_from_trigger'].notna()
    print(f"  EOD mediano : {tp.loc[has_rth,'rth_ret_from_trigger'].median():+.1f}%")
    print(f"  EOD media   : {tp.loc[has_rth,'rth_ret_from_trigger'].mean():+.1f}%")
    print(f"  MFE mediano : {tp.loc[has_rth,'rth_mfe_from_trigger'].median():+.1f}%")
    print(f"  Positivos   : {(tp.loc[has_rth,'rth_ret_from_trigger']>0).sum()}/{has_rth.sum()} ({(tp.loc[has_rth,'rth_ret_from_trigger']>0).mean()*100:.0f}% WR)")

## 6. Falsos positivos — ¿qué son los que disparan pero no son bursts?

In [ ]:
print(f"FALSE POSITIVES: {len(pm_fp)} ticker-days")
print(f"(Trigger pre-market activado pero NO llegaron a Top Gainers >=15%)\n")

if len(pm_fp) > 0:
    # Ver el RTH de los FP para entender si son near-miss o ruido puro
    fp = pm_fp.copy()

    # Añadir RTH data
    rth_eod = rth_bars.groupby(['ticker','date']).agg(
        rth_open=('open','first'),
        rth_close=('close','last'),
        rth_high=('high','max'),
        rth_vol_total=('volume','sum')
    ).reset_index()
    fp = fp.merge(rth_eod, on=['ticker','date'], how='left')
    fp['rth_eod_chg'] = (fp['rth_close'] - fp['prev_close']) / fp['prev_close'] * 100

    print(f"RTH EOD change de los FP:")
    print(f"  Mediana: {fp['rth_eod_chg'].median():+.1f}%")
    print(f"  >15% (near-miss burst): {(fp['rth_eod_chg']>15).sum()}")
    print(f"  5-15%: {((fp['rth_eod_chg']>=5)&(fp['rth_eod_chg']<15)).sum()}")
    print(f"  0-5%: {((fp['rth_eod_chg']>=0)&(fp['rth_eod_chg']<5)).sum()}")
    print(f"  <0% (reversion): {(fp['rth_eod_chg']<0).sum()}")

    print(f"\nDetalle FP (prime 15):")
    for _, r in fp.head(15).iterrows():
        eod = r['rth_eod_chg']
        print(f"  {r['ticker']:<6} {r['date']}  trigger={r['time_trigger']}  vol={r['vol_ratio']:.0f}x  RTH_EOD={eod:+.1f}%")
else:
    print("Sin falsos positivos!")

## 7. Gap pre-market vs continuación RTH

In [ ]:
if len(pm_tp) < 3:
    print("Insuficientes TPs")
else:
    tp = pm_tp.copy()
    has_rth = tp['rth_ret_from_trigger'].notna() & tp['rth_gap_pct'].notna()

    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Pre-Market Scanner  |  vol>{PM_VOL_THRESHOLD}x  price>{PM_PRICE_THRESHOLD}%  '
                 f'|  TP={len(tp)}  FP={len(pm_fp)}  Recall={len(tp)/n_bursts_pm*100:.0f}%', fontsize=12)

    # 1. Hora de detección
    ax = axes[0,0]
    tp['hour_trigger_num'] = tp['time_trigger'].str[:2].astype(float) + tp['time_trigger'].str[3:5].astype(float)/60
    ax.hist(tp['hour_trigger_num'], bins=33, range=(4, 9.5), color='steelblue', alpha=0.8, edgecolor='white')
    ax.axvline(9.5, color='red', linewidth=2, linestyle='--', label='RTH 9:30')
    ax.set_xlabel('Hora ET')
    ax.set_ylabel('N bursts detectados')
    ax.set_title('¿A qué hora dispara el scanner PM?')
    ax.set_xticks([4,5,6,7,8,9,9.5])
    ax.set_xticklabels(['4:00','5:00','6:00','7:00','8:00','9:00','9:30'])
    ax.legend(fontsize=8)

    # 2. Gap primer bar PM vs gap RTH open
    ax = axes[0,1]
    if has_rth.sum() > 3:
        ax.scatter(tp.loc[has_rth,'gap_first_pm_pct'], tp.loc[has_rth,'rth_gap_pct'],
                   alpha=0.6, s=30, color='green')
        ax.plot([0,300],[0,300],'r--',linewidth=1,label='PM gap = RTH gap')
        ax.set_xlabel('Gap primera barra PM vs prev_close (%)')
        ax.set_ylabel('Gap apertura RTH vs prev_close (%)')
        ax.set_title('Gap PM vs Gap RTH open')
        ax.legend(fontsize=8)

    # 3. RTH EOD return desde precio trigger PM
    ax = axes[0,2]
    if has_rth.sum() > 3:
        ax.hist(tp.loc[has_rth,'rth_ret_from_trigger'], bins=20, color='green', alpha=0.8, edgecolor='white')
        ax.axvline(0, color='red', linewidth=1.5, linestyle='--')
        med = tp.loc[has_rth,'rth_ret_from_trigger'].median()
        ax.axvline(med, color='black', linewidth=1.5, label=f'Median={med:+.1f}%')
        ax.set_xlabel('RTH EOD return desde precio trigger PM (%)')
        ax.set_title('Retorno RTH desde entrada PM')
        ax.legend(fontsize=8)

    # 4. Gap PM vs RTH EOD return
    ax = axes[1,0]
    if has_rth.sum() > 3:
        ax.scatter(tp.loc[has_rth,'gap_first_pm_pct'], tp.loc[has_rth,'rth_ret_from_trigger'],
                   alpha=0.6, s=30, color='orange')
        ax.axhline(0, color='red', linewidth=1, linestyle='--')
        ax.set_xlabel('Gap primera barra PM (%)')
        ax.set_ylabel('RTH EOD return desde precio trigger (%)')
        ax.set_title('¿Gap mayor → mejor RTH?')

    # 5. Vol ratio en trigger PM
    ax = axes[1,1]
    ax.hist(tp['vol_ratio'].dropna(), bins=20, color='orange', alpha=0.8, edgecolor='white')
    ax.axvline(PM_VOL_THRESHOLD, color='red', linewidth=1.5, linestyle='--', label=f'Threshold={PM_VOL_THRESHOLD}x')
    ax.set_xlabel('Vol ratio en trigger (vol_bar / avg_vol_20d)')
    ax.set_title('Fuerza del vol spike pre-market')
    ax.legend(fontsize=8)

    # 6. Distribución MFE RTH desde trigger PM
    ax = axes[1,2]
    if has_rth.sum() > 3:
        mfe = tp.loc[has_rth,'rth_mfe_from_trigger']
        ax.hist(mfe, bins=20, color='purple', alpha=0.8, edgecolor='white')
        ax.axvline(mfe.median(), color='red', linewidth=1.5, label=f'Median MFE={mfe.median():+.1f}%')
        ax.set_xlabel('MFE RTH desde precio trigger PM (%)')
        ax.set_title('MFE disponible en RTH')
        ax.legend(fontsize=8)

    plt.tight_layout()
    Path('figures').mkdir(exist_ok=True)
    fig.savefig('figures/premarket_scanner.png', dpi=130, bbox_inches='tight')
    plt.show()
    print("Guardado en figures/premarket_scanner.png")

## 8. Sweep de parámetros PM

In [ ]:
sweep_rows = []
for vol_t in [2, 3, 5, 8, 10]:
    for price_t in [1, 2, 3, 5]:
        res = []
        for _, row in pm_pairs.iterrows():
            r = replay_premarket(row['ticker'], row['date'], pm_bars, rth_bars, daily_idx, float_cache,
                                 vol_threshold=vol_t, price_threshold=price_t, float_max=None)
            if r: res.append(r)
        if not res: continue
        df_ = pd.DataFrame(res)
        triggered_ = df_[df_['triggered']==True]
        if triggered_.empty: continue
        t_fv = triggered_.merge(bursts[['ticker','date','mso_finviz']], on=['ticker','date'], how='left')
        tp_ = t_fv[t_fv['mso_finviz'].notna()]
        fp_ = t_fv[t_fv['mso_finviz'].isna()]
        prec = len(tp_)/len(t_fv)*100 if len(t_fv)>0 else 0
        rec  = len(tp_)/n_bursts_pm*100 if n_bursts_pm>0 else 0
        # Hora mediana de detección para los TP
        tp_times = tp_['mso_pm_trigger'].dropna()
        mins_before = (9*60+30 - 4*60) - tp_times.median() if len(tp_times)>0 else None
        sweep_rows.append({
            'vol_t': vol_t, 'price_t': price_t,
            'triggered': len(triggered_), 'TP': len(tp_), 'FP': len(fp_),
            'precision': round(prec,1), 'recall': round(rec,1),
            'mins_before_open': round(mins_before) if mins_before else None,
        })

sw = pd.DataFrame(sweep_rows)
print("SWEEP PARÁMETROS PRE-MARKET")
print("=" * 75)
print(f"{'vol_t':>6} {'price_t':>8} {'triggered':>10} {'TP':>5} {'FP':>5} {'precision':>10} {'recall':>8} {'min_antes_RTH':>14}")
print("-" * 75)
for _, r in sw.sort_values(['precision','recall'], ascending=[False,False]).iterrows():
    mb = f"{r['mins_before_open']:.0f}" if r['mins_before_open'] else 'N/A'
    print(f"{r['vol_t']:>6.0f} {r['price_t']:>8.0f} {r['triggered']:>10.0f} {r['TP']:>5.0f} "
          f"{r['FP']:>5.0f} {r['precision']:>9.1f}% {r['recall']:>7.1f}% {mb:>14}")

best = sw[(sw['precision'] >= 70) & (sw['recall'] >= 30)]
print()
if not best.empty:
    print("CONFIGS precision>=70% Y recall>=30%:")
    print(best.to_string(index=False))
else:
    sw['score'] = sw['precision']*0.5 + sw['recall']*0.5
    print("Top 5 configuraciones (balance precision/recall):")
    print(sw.nlargest(5,'score').to_string(index=False))

## 9. Comparativa RTH scanner vs Pre-market scanner

In [ ]:
# Cuántos bursts captura PM que RTH no captura y viceversa
rth_db_path = Path('/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_intraday_cache.db')

# Cargar RTH scanner results (replicar con vol>8x, price>4% que ya teníamos)
# Usar barras RTH que ya están cargadas
print("Calculando cobertura combinada PM + RTH scanner...\n")

# Bursts capturados por PM scanner (con params actuales)
pm_tp_set = set(zip(pm_tp['ticker'], pm_tp['date'])) if len(pm_tp) > 0 else set()

# Bursts capturados por RTH scanner (replay rápido vol>8x, price>4%)
from pathlib import Path as _P
rth_scan_results = []
rth_pairs = rth_bars.groupby(['ticker','date']).size().reset_index()[['ticker','date']]
for _, row in rth_pairs.iterrows():
    t, d = row['ticker'], row['date']
    day = rth_bars[(rth_bars['ticker']==t) & (rth_bars['date']==d)].sort_values('mso')
    try:
        avg_vol = daily_idx.loc[(t, d), 'avg_vol_20d']
    except:
        tkr_d = daily_idx.xs(t, level='ticker') if t in daily_idx.index.get_level_values('ticker') else None
        if tkr_d is None: continue
        before = tkr_d[tkr_d.index < d]
        if before.empty: continue
        avg_vol = before.iloc[-1]['avg_vol_20d']
    if not avg_vol or pd.isna(avg_vol) or avg_vol <= 0: continue
    closes = day['close'].values; vols = day['volume'].values
    for i in range(3, len(day)):
        vr = vols[i]/avg_vol
        pc = (closes[i]-closes[i-3])/closes[i-3]*100 if closes[i-3]>0 else 0
        if vr >= 8.0 and pc >= 4.0:
            rth_scan_results.append({'ticker': t, 'date': d, 'detected_rth': True})
            break

rth_tp_set = set()
for r in rth_scan_results:
    if (r['ticker'], r['date']) in burst_set:
        rth_tp_set.add((r['ticker'], r['date']))

both    = pm_tp_set & rth_tp_set
pm_only = pm_tp_set - rth_tp_set
rth_only= rth_tp_set - pm_tp_set
n_total = len(burst_set)

print(f"Total bursts Finviz: {n_total}")
print(f"  Detectados solo PM scanner  : {len(pm_only):>4} ({len(pm_only)/n_total*100:.0f}%)")
print(f"  Detectados solo RTH scanner : {len(rth_only):>4} ({len(rth_only)/n_total*100:.0f}%)")
print(f"  Detectados por AMBOS        : {len(both):>4} ({len(both)/n_total*100:.0f}%)")
print(f"  Detectados por alguno       : {len(pm_tp_set|rth_tp_set):>4} ({len(pm_tp_set|rth_tp_set)/n_total*100:.0f}%)")
print(f"  No detectados por ninguno   : {n_total - len(pm_tp_set|rth_tp_set):>4} ({(n_total-len(pm_tp_set|rth_tp_set))/n_total*100:.0f}%)")
print()
print(f"  → PM captura {len(pm_only)} bursts que RTH scanner jamás vería (pre-market gaps)")
print(f"  → Cobertura combinada: {len(pm_tp_set|rth_tp_set)/n_total*100:.0f}% del universo burst")